# Whisper WAV 전사 및 REF 비교

WAV를 16 kHz mono로 읽어 Whisper Base 또는 학습한 LoRA 모델로 전사합니다. `manifest.jsonl`에서 같은 WAV의 정답 문장(REF)을 찾고, 예측 문장(HYP)을 TXT로 저장한 뒤 WER/CER를 계산합니다.

In [8]:
# 필요한 경우 처음 한 번만 주석을 제거해 실행하세요.
# %pip install -U torch transformers peft accelerate soundfile scipy jiwer safetensors

In [9]:
import json
import math
import re
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import soundfile as sf
import torch
from jiwer import cer, wer
from peft import PeftModel
from scipy.signal import resample_poly
from transformers import WhisperForConditionalGeneration, WhisperProcessor

## 1. 경로와 모델 설정

학습한 LoRA 결과를 평가하려면 `USE_LORA=True`를 사용합니다. 아직 adapter가 없다면 `False`로 바꾸면 원본 Whisper Base를 평가합니다.

In [10]:
WAV_PATH = Path(
    '/home/lmh/project/whisper/summer_bootcamp/minhyeok/'
    'whisper_preprocessed/audio/'
    '000_A0051_S0001_0_G0101_chunk_00019.wav'
)
MANIFEST_PATH = Path(
    '/home/lmh/project/whisper/summer_bootcamp/minhyeok/'
    'whisper_preprocessed/manifest.jsonl'
)

BASE_MODEL_ID = 'openai/whisper-small'
USE_LORA = False
LORA_ADAPTER_DIR = Path(
    '/home/lmh/project/whisper/summer_bootcamp/minhyeok/'
    'whisper_base_lora_output/final_adapter'
)

LANGUAGE = 'korean'
TASK = 'transcribe'
SAMPLE_RATE = 16_000
MAX_NEW_TOKENS = 225
OUTPUT_TXT = WAV_PATH.with_name(WAV_PATH.stem + '_whisper.txt')

if not WAV_PATH.is_file():
    raise FileNotFoundError(f'WAV 파일이 없습니다: {WAV_PATH}')
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'manifest 파일이 없습니다: {MANIFEST_PATH}')
if USE_LORA and not (LORA_ADAPTER_DIR / 'adapter_config.json').is_file():
    raise FileNotFoundError(
        f'LoRA adapter_config.json이 없습니다: {LORA_ADAPTER_DIR}\n'
        '아직 LoRA 학습 전이라면 USE_LORA=False로 설정하세요.'
    )

print('WAV:', WAV_PATH)
print('manifest:', MANIFEST_PATH)
print('model:', 'Whisper Base + LoRA' if USE_LORA else 'Whisper Base')
print('prediction txt:', OUTPUT_TXT)

WAV: /home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_preprocessed/audio/000_A0051_S0001_0_G0101_chunk_00019.wav
manifest: /home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_preprocessed/manifest.jsonl
model: Whisper Base
prediction txt: /home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_preprocessed/audio/000_A0051_S0001_0_G0101_chunk_00019_whisper.txt


## 2. manifest에서 해당 WAV의 REF 찾기

정확한 `audio_path` 일치를 가장 우선하며, 상대경로, 파일명, `id`, `file_id`, `feature_path` 순서로 보조 검색합니다.

In [11]:
def read_jsonl(path: Path) -> List[dict]:
    records: List[dict] = []
    with path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f'{path}:{line_number}: 잘못된 JSON입니다.'
                ) from exc
            record = dict(record)
            record['_manifest_line'] = line_number
            records.append(record)
    if not records:
        raise ValueError(f'manifest가 비어 있습니다: {path}')
    return records


def resolved(path: Path) -> Path:
    return path.expanduser().resolve(strict=False)


def match_score(record: dict, wav_path: Path, manifest_path: Path) -> int:
    target = resolved(wav_path)
    target_name = wav_path.name
    target_stem = wav_path.stem
    path_keys = (
        'audio_path', 'wav_path', 'audio_filepath', 'path', 'audio', 'file'
    )

    # manifest 기준 상대경로 또는 절대경로가 실제 WAV와 정확히 같은 경우
    for key in path_keys:
        raw = record.get(key)
        if not raw or not isinstance(raw, (str, Path)):
            continue
        item_path = Path(str(raw)).expanduser()
        candidate = (
            item_path if item_path.is_absolute()
            else manifest_path.parent / item_path
        )
        if resolved(candidate) == target:
            return 100

    # 전처리 manifest의 id는 일반적으로 WAV stem과 같습니다.
    for key in ('id', 'file_id'):
        value = record.get(key)
        if value is not None and str(value) == target_stem:
            return 90

    # manifest가 다른 위치로 이동한 경우 파일명으로 비교
    for key in path_keys:
        raw = record.get(key)
        if raw and isinstance(raw, (str, Path)):
            if Path(str(raw)).name == target_name:
                return 80

    feature_path = record.get('feature_path')
    if feature_path and Path(str(feature_path)).stem == target_stem:
        return 70
    return -1


def find_reference(
    records: List[dict], wav_path: Path, manifest_path: Path
) -> Tuple[dict, int]:
    scored = [
        (match_score(record, wav_path, manifest_path), record)
        for record in records
    ]
    best_score = max(score for score, _ in scored)
    if best_score < 0:
        raise LookupError(
            'manifest에서 WAV에 해당하는 레코드를 찾지 못했습니다.\n'
            f'WAV stem: {wav_path.stem}'
        )
    best = [record for score, record in scored if score == best_score]
    if len(best) != 1:
        candidates = [
            {
                'line': item.get('_manifest_line'),
                'id': item.get('id', item.get('file_id')),
                'audio_path': item.get('audio_path'),
            }
            for item in best
        ]
        raise LookupError(
            '동일 점수의 manifest 레코드가 여러 개입니다: '
            + json.dumps(candidates, ensure_ascii=False)
        )
    if not str(best[0].get('text', '')).strip():
        raise ValueError('찾은 manifest 레코드의 text가 비어 있습니다.')
    return best[0], best_score


records = read_jsonl(MANIFEST_PATH)
matched_record, reference_match_score = find_reference(
    records, WAV_PATH, MANIFEST_PATH
)
REF = str(matched_record['text']).strip()

print('manifest records:', len(records))
print('matched line:', matched_record['_manifest_line'])
print('match score:', reference_match_score)
print('matched id:', matched_record.get('id', matched_record.get('file_id')))
print('REF:', REF)

manifest records: 31
matched line: 20
match score: 100
matched id: 000_A0051_S0001_0_G0101_chunk_00019
REF: 뭔가 더 자기네들은 매출은 오르는 느낌인 거고 그치. 그치. 이게 그- 어- 어- 경제적으로 보면 어- 어떻게 보면은 한 쪽은 내려가지만 어- 어느 쪽은 올라갈 수 밖에 없는 경우인 거잖아. 경제라는 게 어떻게 비- 그런 상황이고 뭐 대체재라는 게 있- 있고 그러니까 그래서 일단은


## 3. Whisper Base 및 선택적 LoRA adapter 불러오기

In [12]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DTYPE = torch.float16 if DEVICE.type == 'cuda' else torch.float32

processor_source = (
    LORA_ADAPTER_DIR
    if USE_LORA and (LORA_ADAPTER_DIR / 'preprocessor_config.json').is_file()
    else BASE_MODEL_ID
)
processor = WhisperProcessor.from_pretrained(
    str(processor_source), language=LANGUAGE, task=TASK
)

base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)
if USE_LORA:
    model = PeftModel.from_pretrained(
        base_model, LORA_ADAPTER_DIR, is_trainable=False
    )
else:
    model = base_model

model = model.to(DEVICE)
model.eval()

print('device:', DEVICE)
print('dtype:', MODEL_DTYPE)
print('model class:', type(model).__name__)
print('processor source:', processor_source)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1244.60it/s]


device: cuda
dtype: torch.float16
model class: WhisperForConditionalGeneration
processor source: openai/whisper-small


## 4. WAV 전처리, 전사 및 TXT 저장

WAV가 stereo이면 채널 평균으로 mono 변환하고, 16 kHz가 아니면 자동 resampling합니다.

In [13]:
def load_audio_mono_16k(path: Path) -> Tuple[np.ndarray, int]:
    audio_2d, original_sr = sf.read(
        str(path), dtype='float32', always_2d=True
    )
    if audio_2d.size == 0:
        raise ValueError(f'빈 WAV 파일입니다: {path}')

    audio = audio_2d.mean(axis=1).astype(np.float32, copy=False)
    if not np.isfinite(audio).all():
        raise ValueError(f'오디오에 NaN 또는 inf가 있습니다: {path}')

    if original_sr != SAMPLE_RATE:
        common = math.gcd(int(original_sr), SAMPLE_RATE)
        audio = resample_poly(
            audio, SAMPLE_RATE // common, int(original_sr) // common
        ).astype(np.float32, copy=False)

    return audio, int(original_sr)


audio, original_sample_rate = load_audio_mono_16k(WAV_PATH)
duration_seconds = len(audio) / SAMPLE_RATE
if duration_seconds > 30.0:
    print(
        f'주의: 오디오가 {duration_seconds:.3f}초입니다. '
        'Whisper 입력은 앞쪽 최대 30초까지만 사용될 수 있습니다.'
    )

features = processor.feature_extractor(
    audio,
    sampling_rate=SAMPLE_RATE,
    return_tensors='pt',
).input_features.to(device=DEVICE, dtype=MODEL_DTYPE)

with torch.inference_mode():
    predicted_ids = model.generate(
        input_features=features,
        language=LANGUAGE,
        task=TASK,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
    )

HYP = processor.batch_decode(
    predicted_ids, skip_special_tokens=True
)[0].strip()
OUTPUT_TXT.write_text(HYP + '\n', encoding='utf-8')

print('original sample rate:', original_sample_rate)
print('model input sample rate:', SAMPLE_RATE)
print(f'duration: {duration_seconds:.3f}s')
print('input_features:', tuple(features.shape), features.dtype)
print('HYP:', HYP)
print('saved:', OUTPUT_TXT)

[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


original sample rate: 16000
model input sample rate: 16000
duration: 29.550s
input_features: (1, 80, 3000) torch.float16
HYP: 뭔가 더 자기네들은 매출 노르는 느낌인 거고 그치 그치 이게... 경제적으로 보면 어떻게 보면 한쪽은 내려가지만 어느 쪽은 올라갈 수밖에 없는 경우인 거잖아? 경제라는 게 어떻게... 그런 상황이고 대체제 라는 게 있고 그러니까 그래서 일단은
saved: /home/lmh/project/whisper/summer_bootcamp/minhyeok/whisper_preprocessed/audio/000_A0051_S0001_0_G0101_chunk_00019_whisper.txt


## 5. REF와 HYP 비교

원문 점수와 함께 대소문자·문장부호·연속 공백을 정규화한 점수도 출력합니다. 한국어 WER은 띄어쓰기 단위, CER은 문자 단위 오류율입니다.

In [14]:
def normalize_for_score(text: str) -> str:
    text = unicodedata.normalize('NFKC', text).lower()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return ' '.join(text.split())


ref_normalized = normalize_for_score(REF)
hyp_normalized = normalize_for_score(HYP)

raw_wer = 100.0 * wer(REF, HYP)
raw_cer = 100.0 * cer(REF, HYP)
normalized_wer = 100.0 * wer(ref_normalized, hyp_normalized)
normalized_cer = 100.0 * cer(ref_normalized, hyp_normalized)

print('=' * 80)
print('REF:', REF)
print('HYP:', HYP)
print('-' * 80)
print('normalized REF:', ref_normalized)
print('normalized HYP:', hyp_normalized)
print('-' * 80)
print(f'raw WER:        {raw_wer:.2f}%')
print(f'raw CER:        {raw_cer:.2f}%')
print(f'normalized WER: {normalized_wer:.2f}%')
print(f'normalized CER: {normalized_cer:.2f}%')
print('=' * 80)

REF: 뭔가 더 자기네들은 매출은 오르는 느낌인 거고 그치. 그치. 이게 그- 어- 어- 경제적으로 보면 어- 어떻게 보면은 한 쪽은 내려가지만 어- 어느 쪽은 올라갈 수 밖에 없는 경우인 거잖아. 경제라는 게 어떻게 비- 그런 상황이고 뭐 대체재라는 게 있- 있고 그러니까 그래서 일단은
HYP: 뭔가 더 자기네들은 매출 노르는 느낌인 거고 그치 그치 이게... 경제적으로 보면 어떻게 보면 한쪽은 내려가지만 어느 쪽은 올라갈 수밖에 없는 경우인 거잖아? 경제라는 게 어떻게... 그런 상황이고 대체제 라는 게 있고 그러니까 그래서 일단은
--------------------------------------------------------------------------------
normalized REF: 뭔가 더 자기네들은 매출은 오르는 느낌인 거고 그치 그치 이게 그 어 어 경제적으로 보면 어 어떻게 보면은 한 쪽은 내려가지만 어 어느 쪽은 올라갈 수 밖에 없는 경우인 거잖아 경제라는 게 어떻게 비 그런 상황이고 뭐 대체재라는 게 있 있고 그러니까 그래서 일단은
normalized HYP: 뭔가 더 자기네들은 매출 노르는 느낌인 거고 그치 그치 이게 경제적으로 보면 어떻게 보면 한쪽은 내려가지만 어느 쪽은 올라갈 수밖에 없는 경우인 거잖아 경제라는 게 어떻게 그런 상황이고 대체제 라는 게 있고 그러니까 그래서 일단은
--------------------------------------------------------------------------------
raw WER:        47.73%
raw CER:        21.02%
normalized WER: 36.36%
normalized CER: 15.65%
